In [0]:
spark

In [0]:
# Project: Databricks NYC Taxi Lakehouse Analytics
# Notebook: 03_business_metrics_gold
# Objective: Create Gold tables with business KPIs and analytics metrics

from pyspark.sql import functions as F

silver_enriched_table = "workspace.default.silver_yellow_taxi_trips_enriched"

df_taxi_gold_base = spark.table(silver_enriched_table)

print("Gold base table loaded successfully.")
print("Rows:", df_taxi_gold_base.count())

Gold base table loaded successfully.
Rows: 2869714


In [0]:
# Create Gold table: daily taxi metrics

df_gold_daily_metrics = (
    df_taxi_gold_base
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy("pickup_date")
)

display(df_gold_daily_metrics)

pickup_date,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
2002-12-31,1,10.5,10.5,6.5,0.63,6.03
2009-01-01,3,127.69,42.56,33.33,7.44,27.62
2023-12-31,10,224.62,22.46,14.41,2.6,10.16
2024-01-01,75529,2338847.47,30.97,22.2,4.66,16.66
2024-01-02,73007,2256011.47,30.9,21.39,4.21,17.04
2024-01-03,79958,2338629.52,29.25,20.07,3.95,16.65
2024-01-04,100149,2781199.67,27.77,18.77,3.36,16.14
2024-01-05,100138,2701994.39,26.98,18.15,3.83,15.46
2024-01-06,93769,2404157.93,25.64,17.66,3.2,14.7
2024-01-07,65224,1872602.09,28.71,19.84,3.96,14.01


In [0]:
# Filter Gold base for valid January 2024 records only

df_taxi_gold_base_valid = (
    df_taxi_gold_base
    .filter(F.col("pickup_date") >= F.lit("2024-01-01"))
    .filter(F.col("pickup_date") <= F.lit("2024-01-31"))
)

print("Original Gold base rows:", df_taxi_gold_base.count())
print("Valid January 2024 rows:", df_taxi_gold_base_valid.count())

Original Gold base rows: 2869714
Valid January 2024 rows: 2869697


In [0]:
# Create Gold table: daily taxi metrics with valid dates

df_gold_daily_metrics = (
    df_taxi_gold_base_valid
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy("pickup_date")
)

display(df_gold_daily_metrics)

pickup_date,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
2024-01-01,75529,2338847.47,30.97,22.2,4.66,16.66
2024-01-02,73007,2256011.47,30.9,21.39,4.21,17.04
2024-01-03,79958,2338629.52,29.25,20.07,3.95,16.65
2024-01-04,100149,2781199.67,27.77,18.77,3.36,16.14
2024-01-05,100138,2701994.39,26.98,18.15,3.83,15.46
2024-01-06,93769,2404157.93,25.64,17.66,3.2,14.7
2024-01-07,65224,1872602.09,28.71,19.84,3.96,14.01
2024-01-08,77744,2200822.01,28.31,19.09,3.57,15.73
2024-01-09,90344,2324271.17,25.73,17.23,3.95,15.16
2024-01-10,92519,2528753.08,27.33,18.21,3.41,15.29


In [0]:
# Save Gold Delta table: daily taxi metrics

gold_daily_table = "workspace.default.gold_daily_taxi_metrics"

df_gold_daily_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_daily_table)

print("Gold daily taxi metrics table created successfully.")

Gold daily taxi metrics table created successfully.


In [0]:
# Validate Gold Delta table: daily taxi metrics

gold_daily_count = spark.table("workspace.default.gold_daily_taxi_metrics").count()

print("Gold daily rows:", gold_daily_count)

display(
    spark.table("workspace.default.gold_daily_taxi_metrics")
    .orderBy("pickup_date")
)

Gold daily rows: 31


pickup_date,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
2024-01-01,75529,2338847.47,30.97,22.2,4.66,16.66
2024-01-02,73007,2256011.47,30.9,21.39,4.21,17.04
2024-01-03,79958,2338629.52,29.25,20.07,3.95,16.65
2024-01-04,100149,2781199.67,27.77,18.77,3.36,16.14
2024-01-05,100138,2701994.39,26.98,18.15,3.83,15.46
2024-01-06,93769,2404157.93,25.64,17.66,3.2,14.7
2024-01-07,65224,1872602.09,28.71,19.84,3.96,14.01
2024-01-08,77744,2200822.01,28.31,19.09,3.57,15.73
2024-01-09,90344,2324271.17,25.73,17.23,3.95,15.16
2024-01-10,92519,2528753.08,27.33,18.21,3.41,15.29


In [0]:
# Create Gold table: pickup borough metrics

df_gold_pickup_borough_metrics = (
    df_taxi_gold_base_valid
    .groupBy("pickup_borough")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy(F.desc("total_trips"))
)

display(df_gold_pickup_borough_metrics)

pickup_borough,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
Manhattan,2573386,5.857773114E7,22.76,14.92,2.72,13.81
Queens,257799,1.862508759E7,72.25,52.8,12.82,33.2
Brooklyn,22282,740002.93,33.21,28.78,14.81,32.34
Unknown,9638,263658.4,27.36,18.99,3.31,16.13
Bronx,5756,208380.3,36.2,33.11,7.83,39.54
N/A,741,59080.88,79.73,69.75,9.36,24.07
EWR,49,4860.04,99.18,84.37,5.77,7.71
Staten Island,46,2817.96,61.26,46.14,10.02,28.79


In [0]:
# Create Gold table: pickup borough metrics excluding Unknown and N/A

df_gold_pickup_borough_metrics = (
    df_taxi_gold_base_valid
    .filter(~F.col("pickup_borough").isin("Unknown", "N/A"))
    .groupBy("pickup_borough")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy(F.desc("total_trips"))
)

display(df_gold_pickup_borough_metrics)

pickup_borough,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
Manhattan,2573386,5.857773114E7,22.76,14.92,2.72,13.81
Queens,257799,1.862508759E7,72.25,52.8,12.82,33.2
Brooklyn,22282,740002.93,33.21,28.78,14.81,32.34
Bronx,5756,208380.3,36.2,33.11,7.83,39.54
EWR,49,4860.04,99.18,84.37,5.77,7.71
Staten Island,46,2817.96,61.26,46.14,10.02,28.79


In [0]:
# Save Gold Delta table: pickup borough metrics

gold_pickup_borough_table = "workspace.default.gold_pickup_borough_metrics"

df_gold_pickup_borough_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_pickup_borough_table)

print("Gold pickup borough metrics table created successfully.")

Gold pickup borough metrics table created successfully.


In [0]:
# Validate Gold pickup borough metrics table

gold_pickup_borough_count = spark.table(
    "workspace.default.gold_pickup_borough_metrics"
).count()

print("Gold pickup borough rows:", gold_pickup_borough_count)

display(
    spark.table("workspace.default.gold_pickup_borough_metrics")
    .orderBy(F.desc("total_trips"))
)

Gold pickup borough rows: 6


pickup_borough,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
Manhattan,2573386,5.857773114E7,22.76,14.92,2.72,13.81
Queens,257799,1.862508759E7,72.25,52.8,12.82,33.2
Brooklyn,22282,740002.93,33.21,28.78,14.81,32.34
Bronx,5756,208380.3,36.2,33.11,7.83,39.54
EWR,49,4860.04,99.18,84.37,5.77,7.71
Staten Island,46,2817.96,61.26,46.14,10.02,28.79


In [0]:
# Create Gold table: hourly taxi metrics

df_gold_hourly_metrics = (
    df_taxi_gold_base_valid
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy("pickup_hour")
)

display(df_gold_hourly_metrics)

pickup_hour,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
0,75237,2150345.44,28.58,19.7,3.86,14.97
1,50481,1307733.17,25.91,17.74,3.27,14.05
2,34961,860195.56,24.6,16.63,3.04,13.37
3,22942,612920.07,26.72,18.54,3.52,13.6
4,15276,496201.28,32.48,23.45,4.87,15.71
5,17494,657081.1,37.56,27.5,9.18,17.03
6,39415,1188536.45,30.15,22.03,13.47,16.34
7,80858,2147049.58,26.55,18.77,6.18,15.56
8,113484,2899041.9,25.55,17.84,5.59,15.73
9,125600,3253677.78,25.91,17.96,3.05,15.81


In [0]:
# Save Gold Delta table: hourly taxi metrics

gold_hourly_table = "workspace.default.gold_hourly_taxi_metrics"

df_gold_hourly_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_hourly_table)

print("Gold hourly taxi metrics table created successfully.")

Gold hourly taxi metrics table created successfully.


In [0]:
# Validate Gold hourly taxi metrics table

gold_hourly_count = spark.table(
    "workspace.default.gold_hourly_taxi_metrics"
).count()

print("Gold hourly rows:", gold_hourly_count)

display(
    spark.table("workspace.default.gold_hourly_taxi_metrics")
    .orderBy("pickup_hour")
)

Gold hourly rows: 24


pickup_hour,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
0,75237,2150345.44,28.58,19.7,3.86,14.97
1,50481,1307733.17,25.91,17.74,3.27,14.05
2,34961,860195.56,24.6,16.63,3.04,13.37
3,22942,612920.07,26.72,18.54,3.52,13.6
4,15276,496201.28,32.48,23.45,4.87,15.71
5,17494,657081.1,37.56,27.5,9.18,17.03
6,39415,1188536.45,30.15,22.03,13.47,16.34
7,80858,2147049.58,26.55,18.77,6.18,15.56
8,113484,2899041.9,25.55,17.84,5.59,15.73
9,125600,3253677.78,25.91,17.96,3.05,15.81


In [0]:
# Save Gold Delta table: hourly taxi metrics

gold_hourly_table = "workspace.default.gold_hourly_taxi_metrics"

df_gold_hourly_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_hourly_table)

print("Gold hourly taxi metrics table created successfully.")

Gold hourly taxi metrics table created successfully.


In [0]:
# Validate Gold hourly taxi metrics table

gold_hourly_count = spark.table(
    "workspace.default.gold_hourly_taxi_metrics"
).count()

print("Gold hourly rows:", gold_hourly_count)

display(
    spark.table("workspace.default.gold_hourly_taxi_metrics")
    .orderBy("pickup_hour")
)

Gold hourly rows: 24


pickup_hour,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
0,75237,2150345.44,28.58,19.7,3.86,14.97
1,50481,1307733.17,25.91,17.74,3.27,14.05
2,34961,860195.56,24.6,16.63,3.04,13.37
3,22942,612920.07,26.72,18.54,3.52,13.6
4,15276,496201.28,32.48,23.45,4.87,15.71
5,17494,657081.1,37.56,27.5,9.18,17.03
6,39415,1188536.45,30.15,22.03,13.47,16.34
7,80858,2147049.58,26.55,18.77,6.18,15.56
8,113484,2899041.9,25.55,17.84,5.59,15.73
9,125600,3253677.78,25.91,17.96,3.05,15.81


In [0]:
# Create Gold table: top pickup zones metrics

df_gold_pickup_zone_metrics = (
    df_taxi_gold_base_valid
    .filter(~F.col("pickup_borough").isin("Unknown", "N/A"))
    .filter(F.col("pickup_zone").isNotNull())
    .groupBy("pickup_borough", "pickup_zone")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
    )
    .orderBy(F.desc("total_trips"))
)

display(df_gold_pickup_zone_metrics.limit(20))

pickup_borough,pickup_zone,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
Manhattan,Midtown Center,140140,3357268.7,23.96,15.5,2.6,14.62
Manhattan,Upper East Side South,140120,2772039.65,19.78,12.38,1.72,11.63
Queens,JFK Airport,138450,1.119825616E7,80.88,62.87,15.94,38.1
Manhattan,Upper East Side North,133961,2714041.21,20.26,12.86,1.87,11.89
Manhattan,Midtown East,104345,2434416.56,23.33,15.07,2.26,14.06
Manhattan,Times Sq/Theatre District,102958,2766396.27,26.87,17.9,2.97,16.06
Manhattan,Penn Station/Madison Sq West,102152,2462572.23,24.11,16.09,2.3,15.94
Manhattan,Lincoln Square East,101794,2171532.03,21.33,13.62,2.12,12.96
Queens,LaGuardia Airport,87694,5820788.27,66.38,42.35,9.71,27.57
Manhattan,Upper West Side South,86468,1834602.47,21.22,13.61,2.29,12.17


In [0]:
# Save Gold Delta table: pickup zone metrics

gold_pickup_zone_table = "workspace.default.gold_pickup_zone_metrics"

df_gold_pickup_zone_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_pickup_zone_table)

print("Gold pickup zone metrics table created successfully.")

Gold pickup zone metrics table created successfully.


In [0]:
# Validate Gold pickup zone metrics table

gold_pickup_zone_count = spark.table(
    "workspace.default.gold_pickup_zone_metrics"
).count()

print("Gold pickup zone rows:", gold_pickup_zone_count)

display(
    spark.table("workspace.default.gold_pickup_zone_metrics")
    .orderBy(F.desc("total_trips"))
    .limit(20)
)

Gold pickup zone rows: 254


pickup_borough,pickup_zone,total_trips,total_revenue,avg_total_amount,avg_fare_amount,avg_trip_distance,avg_trip_duration_minutes
Manhattan,Midtown Center,140140,3357268.7,23.96,15.5,2.6,14.62
Manhattan,Upper East Side South,140120,2772039.65,19.78,12.38,1.72,11.63
Queens,JFK Airport,138450,1.119825616E7,80.88,62.87,15.94,38.1
Manhattan,Upper East Side North,133961,2714041.21,20.26,12.86,1.87,11.89
Manhattan,Midtown East,104345,2434416.56,23.33,15.07,2.26,14.06
Manhattan,Times Sq/Theatre District,102958,2766396.27,26.87,17.9,2.97,16.06
Manhattan,Penn Station/Madison Sq West,102152,2462572.23,24.11,16.09,2.3,15.94
Manhattan,Lincoln Square East,101794,2171532.03,21.33,13.62,2.12,12.96
Queens,LaGuardia Airport,87694,5820788.27,66.38,42.35,9.71,27.57
Manhattan,Upper West Side South,86468,1834602.47,21.22,13.61,2.29,12.17


In [0]:
# Validate all Gold tables created in this project

gold_tables = [
    "workspace.default.gold_daily_taxi_metrics",
    "workspace.default.gold_pickup_borough_metrics",
    "workspace.default.gold_hourly_taxi_metrics",
    "workspace.default.gold_pickup_zone_metrics"
]

for table in gold_tables:
    count_rows = spark.table(table).count()
    print(f"{table}: {count_rows} rows")

workspace.default.gold_daily_taxi_metrics: 31 rows
workspace.default.gold_pickup_borough_metrics: 6 rows
workspace.default.gold_hourly_taxi_metrics: 24 rows
workspace.default.gold_pickup_zone_metrics: 254 rows
